In [1]:
import glob,os,shutil

In [2]:
infiles = glob.glob(r"J:\BLA_four_types\plots\*")

In [3]:
for infile in infiles:

     
    newpath_tmp = infile[0:-6]+"soma.jpg"
    newpath = newpath_tmp.replace("soma_fig","soma")
    shutil.copy(infile,newpath)

In [ ]:
 
newpath_tmp = infile[0:-6]+"soma.jpg"
newpath = newpath_tmp.replace("soma_fig","soma")


newpath = infile[0:]

In [ ]:
aa = "E:\allen_swc\tesr\1060693328_17302_2634-X14565-Y41960_reg_ec_489_soma.png"

In [13]:
from pathlib import Path
from PIL import Image, ImageChops

def crop_margins(img, padding=2):
    """自动裁剪图片周围的空白/白色边框，使主体更紧凑"""
    bg = Image.new(img.mode, img.size, (255, 255, 255))
    diff = ImageChops.difference(img, bg)
    bbox = diff.getbbox()
    if bbox:
        left, upper, right, lower = bbox
        left = max(0, left - padding)
        upper = max(0, upper - padding)
        right = min(img.width, right + padding)
        lower = min(img.height, lower + padding)
        return img.crop((left, upper, right, lower))
    return img

def merge_one_set(soma_path, overlap_px=10):
    """
    soma_path: soma图片路径
    overlap_px: 重叠/紧密程度，值越大图片靠得越紧（可设为 0 ~ 30）
    """
    folder = soma_path.parent
    stem = soma_path.name.replace("_soma.jpg", "").replace("_soma.png", "")
    print(f"Processing: {stem}")
    
    stem_new = stem[0:10] + "_" + stem.split("_")[2]

    # ---- 自动推断3个视角图 ----
    front = folder / f"{stem_new}_front.png"
    left  = folder / f"{stem_new}_left.png"
    top   = folder / f"{stem_new}_top.png"

    views = [front, left, top]

    # —— 检查文件是否齐全 ——
    missing = [p.name for p in views if not p.exists()]
    if missing:
        print("❌ 缺失视角图，跳过:", soma_path.name)
        print("   missing:", missing)
        return

    # ---- 1. 读图并裁切边缘空白 ----
    soma = crop_margins(Image.open(soma_path).convert("RGB"))
    img_front = crop_margins(Image.open(front).convert("RGB"))
    img_left = crop_margins(Image.open(left).convert("RGB"))
    img_top = crop_margins(Image.open(top).convert("RGB"))

    # ---- 2. 以 soma 的高度为基准缩放另外 3 张图 ----
    soma_h = soma.height

    def resize_to_height(img, target_h):
        aspect = img.width / img.height
        new_w = int(target_h * aspect)
        return img.resize((new_w, target_h), Image.Resampling.LANCZOS)

    img_front = resize_to_height(img_front, soma_h)
    img_left  = resize_to_height(img_left, soma_h)
    img_top   = resize_to_height(img_top, soma_h)

    # 排布布局：
    # 第一行：soma (左上) | front (右上)
    # 第二行：left (左下) | top (右下)

    # ---- 3. 计算对齐后的网格列宽与行高（考虑 overlap） ----
    col0_w = max(soma.width, img_left.width)
    col1_w = max(img_front.width, img_top.width)

    # 包含重叠/缩进的总尺寸
    canvas_w = col0_w + col1_w - overlap_px
    canvas_h = soma_h * 2 - overlap_px

    canvas = Image.new("RGB", (canvas_w, canvas_h), (255, 255, 255))

    # ---- 4. 粘贴第一行（考虑水平 & 垂直重叠） ----
    # 左上: soma
    canvas.paste(soma, (0, 0))
    # 右上: front
    x_front = col0_w - overlap_px
    canvas.paste(img_front, (x_front, 0))

    # ---- 5. 粘贴第二行（考虑水平 & 垂直重叠） ----
    y_row2 = soma_h - overlap_px
    # 左下: left
    canvas.paste(img_left, (0, y_row2))
    # 右下: top
    canvas.paste(img_top, (x_front, y_row2))

    # ---- 6. 保存 ----
    save_name = soma_path.name.replace(".png", "_merge_2x2.png").replace(".jpg", "_merge_2x2.png")
    save_path = folder / save_name
    canvas.save(save_path, dpi=(300, 300))

    print(f"✅ merged: {save_name}\n")


def batch_merge(folder):
    folder = Path(folder)

    soma_list = list(folder.glob("*_soma.png")) + list(folder.glob("*_soma.jpg"))
    print(f"Found soma images: {len(soma_list)}")

    for soma in soma_list:
        # overlap_px 控制图片间交叠/紧凑程度
        # 如果觉得图片重叠到了主要内容，可以调小（如 5 或 0）；如果觉得间隙还大，可以调大（如 15 或 20）
        merge_one_set(soma, overlap_px=10)


# ===============================
if __name__ == "__main__":
    batch_merge(r"J:\BLA_four_types\plots")

Found soma images: 184
Processing: 221058_040_Sst_BLAa_726
✅ merged: 221058_040_Sst_BLAa_726_soma_merge_2x2.png

Processing: 221058_042_Sst_BLAa_737
✅ merged: 221058_042_Sst_BLAa_737_soma_merge_2x2.png

Processing: 221058_071_Sst_BLAa_713
✅ merged: 221058_071_Sst_BLAa_713_soma_merge_2x2.png

Processing: 221058_088_Sst_BLAa_714
✅ merged: 221058_088_Sst_BLAa_714_soma_merge_2x2.png

Processing: 221058_104_Sst_BLAa_685
✅ merged: 221058_104_Sst_BLAa_685_soma_merge_2x2.png

Processing: 221058_107_Sst_BLAa_682
✅ merged: 221058_107_Sst_BLAa_682_soma_merge_2x2.png

Processing: 221058_113_Sst_BLAa_726
✅ merged: 221058_113_Sst_BLAa_726_soma_merge_2x2.png

Processing: 221297_036_Sst_BLAa_675
✅ merged: 221297_036_Sst_BLAa_675_soma_merge_2x2.png

Processing: 221297_037_Sst_BLAa_680
✅ merged: 221297_037_Sst_BLAa_680_soma_merge_2x2.png

Processing: 221297_038_Sst_BLAa_673
✅ merged: 221297_038_Sst_BLAa_673_soma_merge_2x2.png

Processing: 230058_073_Sst_BLAa_691
✅ merged: 230058_073_Sst_BLAa_691_soma_me

In [11]:
from pathlib import Path
from PIL import Image, ImageChops

def crop_margins(img, padding=5):
    """自动裁剪图片周围的空白/白色边框，使主体更紧凑"""
    bg = Image.new(img.mode, img.size, (255, 255, 255))
    diff = ImageChops.difference(img, bg)
    bbox = diff.getbbox()
    if bbox:
        left, upper, right, lower = bbox
        left = max(0, left - padding)
        upper = max(0, upper - padding)
        right = min(img.width, right + padding)
        lower = min(img.height, lower + padding)
        return img.crop((left, upper, right, lower))
    return img

def merge_one_set(soma_path, grid_size=800):
    folder = soma_path.parent
    stem = soma_path.name.replace("_soma.jpg", "").replace("_soma.png", "")
    print(f"Processing: {stem}")
    
    stem_new = stem[0:10] + "_" + stem.split("_")[2]

    # ---- 自动推断3个视角图 ----
    front = folder / f"{stem_new}_front.png"
    left  = folder / f"{stem_new}_left.png"
    top   = folder / f"{stem_new}_top.png"

    views = [front, left, top]

    # —— 检查文件是否齐全 ——
    missing = [p.name for p in views if not p.exists()]
    if missing:
        print("❌ 缺失视角图，跳过:", soma_path.name)
        print("   missing:", missing)
        return

    # ---- 读图并裁切白边 ----
    soma = crop_margins(Image.open(soma_path).convert("RGB"))
    img_front = crop_margins(Image.open(front).convert("RGB"))
    img_left = crop_margins(Image.open(left).convert("RGB"))
    img_top = crop_margins(Image.open(top).convert("RGB"))

    # 按 2x2 顺序排列：左上, 右上, 左下, 右下
    images = [soma, img_front, img_left, img_top]

    # ---- 计算网格尺寸 ----
    # 策略：为了 2x2 整齐划一，我们创建一个固定大小的格子，所有图片等比例缩放后放入格子中心。
    # 我们以所有图片中最大的宽或高来决定单个格子的基准大小
    max_w = max(im.width for im in images)
    max_h = max(im.height for im in images)
    
    # 设定单个格子的尺寸为 (max_w, max_h)，你也可以硬编码比如 cell_w=1000, cell_h=1000
    cell_w = max_w
    cell_h = max_h

    # 创建 2x2 总画布，背景为纯白
    canvas_w = cell_w * 2
    canvas_h = cell_h * 2
    canvas = Image.new("RGB", (canvas_w, canvas_h), (255, 255, 255))

    # ---- 将图片放入 2x2 网格并居中 ----
    positions = [
        (0, 0),             # 左上 (soma)
        (cell_w, 0),        # 右上 (front)
        (0, cell_h),        # 左下 (left)
        (cell_w, cell_h)    # 右下 (top)
    ]

    for img, (x, y) in zip(images, positions):
        # 让图片在对应的格子内居中
        offset_x = x + (cell_w - img.width) // 2
        offset_y = y + (cell_h - img.height) // 2
        canvas.paste(img, (offset_x, offset_y))

    # ---- 保存 ----
    save_name = soma_path.name.replace(".png", "_merge_2x2.png").replace(".jpg", "_merge_2x2.png")
    save_path = folder / save_name
    canvas.save(save_path, dpi=(300, 300))

    print(f"✅ merged: {save_name}\n")


def batch_merge(folder):
    folder = Path(folder)

    soma_list = list(folder.glob("*_soma.png")) + list(folder.glob("*_soma.jpg"))
    print(f"Found soma images: {len(soma_list)}")

    for soma in soma_list[0:3]:
        merge_one_set(soma)


# ===============================
if __name__ == "__main__":
    batch_merge(r"J:\BLA_four_types\plots")

Found soma images: 184
Processing: 221058_040_Sst_BLAa_726
✅ merged: 221058_040_Sst_BLAa_726_soma_merge_2x2.png

Processing: 221058_042_Sst_BLAa_737
✅ merged: 221058_042_Sst_BLAa_737_soma_merge_2x2.png

Processing: 221058_071_Sst_BLAa_713
✅ merged: 221058_071_Sst_BLAa_713_soma_merge_2x2.png



In [ ]:
from pathlib import Path
from PIL import Image
import pandas as pd

def merge_one_set(soma_path):
    folder = soma_path.parent
    stem = soma_path.name.replace("_soma.jpg", "")
    print(stem)
    # stem_new = stem.split("")[0]  # 去掉"部分
    stem_new = stem[0:10]+"_"+stem.split("_")[2]  # 去掉"部分
    

    # 自动推断4个视角图
    front = folder / f"{stem_new}_front.png"
    left  = folder / f"{stem_new}_left.png"
    se    = folder / f"{stem_new}_se.png"
    top   = folder / f"{stem_new}_top.png"

    views = [front, left, se, top]

    # —— 检查文件是否齐全 ——
    missing = [p.name for p in views if not p.exists()]
    if missing:
        print("❌ 缺失视角图，跳过:", soma_path.name)
        print("   missing:", missing)
        return

    # ---- 读图 ----
    soma = Image.open(soma_path).convert("RGB")
    imgs = [Image.open(p).convert("RGB") for p in views]

    # ---- 右侧2×2规格 ----
    soma_h = soma.height
    cell_h = soma_h // 2

    aspect = imgs[0].width / imgs[0].height
    cell_w = int(cell_h * aspect)

    resized = [im.resize((cell_w, cell_h)) for im in imgs]

    total_w = soma.width + cell_w * 2
    total_h = soma_h

    canvas = Image.new("RGB", (total_w, total_h), (255,255,255))

    # ---- 粘贴 ----
    canvas.paste(soma, (0,0))

    positions = [
        (soma.width, 0),
        (soma.width + cell_w, 0),
        (soma.width, cell_h),
        (soma.width + cell_w, cell_h),
    ]

    for im, pos in zip(resized, positions):
        canvas.paste(im, pos)

    # ---- 保存 ----
    save_name = soma_path.name.replace(".png", "merge.png")
    save_path = folder / save_name
    canvas.save(save_path, dpi=(300,300))

    print("✅ merged:", save_name)


def batch_merge(folder):
    folder = Path(folder)

    soma_list = list(folder.glob("*_soma.png"))
    print(f"Found soma images: {len(soma_list)}")

    # list_tmp = df["0"].tolist()
    # soma_list = [Path(item.replace("_merge.png","_soma.jpg")) for item in list_tmp]

    for soma in soma_list[0:3]:
        merge_one_set(soma)


# ===============================
if __name__ == "__main__":
    # df = pd.read_csv(r"J:\BLA_Cck\BLA_swc_list2.csv",index_col=0)
    batch_merge(r"J:\BLA_four_types\plots")


Found soma images: 92
221058_040_Sst_BLAa_726_soma.png
✅ merged: 221058_040_Sst_BLAa_726_somamerge.png
221058_042_Sst_BLAa_737_soma.png
✅ merged: 221058_042_Sst_BLAa_737_somamerge.png
221058_071_Sst_BLAa_713_soma.png
✅ merged: 221058_071_Sst_BLAa_713_somamerge.png


In [ ]:
folder = Path(r"F:\HIP\ACB")

In [ ]:
folder


In [ ]:
import pandas as pd
df = pd.read_csv(r"F:\HIP\ACB\un.csv",index_col=0)

In [ ]:
list_tmp = df["0"].tolist()
somalist = [item.replace("_merge.png","_soma.jpg")for item in list_tmp]

In [ ]:
from pathlib import Path
from PIL import Image

def merge_one_set(soma_path):
    folder = soma_path.parent
    stem = soma_path.name.replace("_soma.jpg", "")
    print(stem)
    # stem_new = stem.split("")[0]  # 去掉"部分
    stem_new = stem  # 去掉"部分
    

    # 自动推断4个视角图
    front = folder / f"{stem_new}_front.png"
    left  = folder / f"{stem_new}_left.png"
    se    = folder / f"{stem_new}_se.png"
    top   = folder / f"{stem_new}_top.png"

    views = [front, left, se, top]

    # —— 检查文件是否齐全 ——
    missing = [p.name for p in views if not p.exists()]
    if missing:
        print("❌ 缺失视角图，跳过:", soma_path.name)
        print("   missing:", missing)
        return

    # ---- 读图 ----
    soma = Image.open(soma_path).convert("RGB")
    imgs = [Image.open(p).convert("RGB") for p in views]

    # ---- 右侧2×2规格 ----
    soma_h = soma.height
    cell_h = soma_h // 2

    aspect = imgs[0].width / imgs[0].height
    cell_w = int(cell_h * aspect)

    resized = [im.resize((cell_w, cell_h)) for im in imgs]

    total_w = soma.width + cell_w * 2
    total_h = soma_h

    canvas = Image.new("RGB", (total_w, total_h), (255,255,255))

    # ---- 粘贴 ----
    canvas.paste(soma, (0,0))

    positions = [
        (soma.width, 0),
        (soma.width + cell_w, 0),
        (soma.width, cell_h),
        (soma.width + cell_w, cell_h),
    ]

    for im, pos in zip(resized, positions):
        canvas.paste(im, pos)

    # ---- 保存 ----
    save_name = soma_path.name.replace("soma.jpg", "merge.png")
    save_path = folder / save_name
    canvas.save(save_path, dpi=(300,300))

    print("✅ merged:", save_name)


def batch_merge(folder):
    folder = Path(folder)

    soma_list = list(folder.glob("*_soma.jpg"))
    print(f"Found soma images: {len(soma_list)}")

    for soma in soma_list[200:500]:
        merge_one_set(soma)


# ===============================
if __name__ == "__main__":
    df = pd.read
    batch_merge(r"F:\HIP\ACB\plot",df)


In [ ]:
neurons = os.listdir(r"E:\allen_swc\soma_include")

In [ ]:
input = r"E:\allen_swc\reg_swc"
output = r"E:\allen_swc\choose_swc"
os.makedirs(output, exist_ok=True)

for neuron in neurons:
    filename1 = neuron.split("g_")[0]+"g.swc"
    infile = os.path.join(input,filename1)
    outfile = os.path.join(output,filename1)
    shutil.copy(infile,outfile)

 

In [ ]:
input = r"E:\allen_swc\plot"
output = r"E:\allen_swc\choe_plot"

for neuron in neurons:
    filename1 = neuron.split("g_")[0]+"g_front.png"
    filename2 = neuron.split("g_")[0]+"g_left.png"
    filename3 = neuron.split("g_")[0]+"g_se.png"
    filename4 = neuron.split("g_")[0]+"g_top.png"

    infile = os.path.join(input,filename1)
    outfile = os.path.join(output,filename1)
    shutil.copy(infile,outfile)

    infile = os.path.join(input,filename2)
    outfile = os.path.join(output,filename2)
    shutil.copy(infile,outfile)

    infile = os.path.join(input,filename3)
    outfile = os.path.join(output,filename3)
    shutil.copy(infile,outfile)

    infile = os.path.join(input,filename4)
    outfile = os.path.join(output,filename4)
    shutil.copy(infile,outfile)
    

In [ ]:
neurons[1].split("g_")+"g_front.png"

In [ ]:
from openpyxl import Workbook

# Create workbook
wb = Workbook()
ws = wb.active
ws.title = "Glucose_2025-12-09"

data = [
    ["MouseID", "CSF_mmol_L", "CSF_time", "Blood_mmol_L", "Blood2_mmol_L", "Blood_time", "BW_g"],
    [249, 8.9, "", 5.9, 9.9, "15:35", 22.55],
    [189, 12.6, "15:57", 7.7, 7.6, "15:49", 21.91],
    [186, 15.4, "16:06", 10.4, 10.5, "16:10", 22.30],
    [190, 12.8, "16:17", 10.8, "", "16:19", 22.98],
    [215, 5.2, "16:29", 6.8, "", "16:41", 21.20],
    [217, 5.3, "16:41", 6.5, "", "16:42", 20.21],
    [215, 7.1, "17:07", 9.0, "", "17:10", 21.80],
    [185, 6.6, "17:12", 5.5, 5.7, "17:15", 22.93],
    [184, 8.5, "17:34", 7.7, "", "17:28", 22.86],
    [184, 8.4, "17:35", 6.4, "", "17:40", 22.98],
    [182, 9.1, "17:44", 7.2, "", "17:52", 22.92],
    [181, 6.8, "18:01", 5.1, "", "18:03", 20.82],
    [191, 8.7, "18:15", 8.1, "", "19:55", ""],
    [212, 10.4, "18:48", 7.6, "", "", ""],
    [187, 8.2, "18:57", 7.5, "", "", ""],
]

for row in data:
    ws.append(row)

file_path = r"C:\Users\zljia\Desktop\123\Glucose_2025-12-09.xlsx"
wb.save(file_path)

file_path


In [ ]:
import os
from PIL import Image
import glob

def get_image_sizes(folder_path):
    """
    获取文件夹中所有图片的大小
    """
    # 支持的图片格式
    image_extensions = ['*.jpg', '*.jpeg', '*.png', '*.gif', '*.bmp', '*.tiff', '*.webp']
    
    print(f"正在扫描文件夹: {folder_path}")
    print("-" * 50)
    
    for extension in image_extensions:
        # 使用通配符匹配所有图片文件
        pattern = os.path.join(folder_path, extension)
        image_files = glob.glob(pattern) + glob.glob(pattern.upper())
        
        for image_path in image_files:
            try:
                with Image.open(image_path) as img:
                    width, height = img.size
                    file_size = os.path.getsize(image_path)  # 文件大小（字节）
                    file_size_mb = file_size / (1024 * 1024)  # 转换为MB
                    
                    print(f"文件名: {os.path.basename(image_path)}")
                    print(f"尺寸: {width} x {height} 像素")
                    print(f"文件大小: {file_size_mb:.2f} MB")
                    print(f"完整路径: {image_path}")
                    print("-" * 30)
                    
            except Exception as e:
                print(f"无法读取图片 {image_path}: {e}")
                print("-" * 30)

# if __name__ == "__main__":
#     # 指定文件夹路径
#     folder_path = input("请输入图片文件夹路径: ").strip()
    
#     # 如果路径为空，使用当前文件夹
#     if not folder_path:
#         folder_path = "."
    
#     if os.path.exists(folder_path):
#         get_image_sizes(folder_path)
#     else:
#         print("指定的路径不存在！")

In [ ]:
get_image_sizes(r"E:\allen_swc\tesr")